# Marker Repo - calculations: Scores, homology

In this notebook, individual markers can be weighted using various functions. In addition, BioMart or the HomoloGene DB enable genes to be transferred from a source organism to a target organism. Thus, even analyses with specific organisms can be performed without already existing suitable lists.

## Loading packages

In [1]:
import markerrepo.marker_repo as mr
import markerrepo.calculations as calc
import markerrepo.utils as utils
import markerrepo.wrappers as wrap

# Calculate scores

This part compares and scores markers from selected marker lists using Ubiquitousness Index.
A score of '0' signifies that the marker is the most specific within this selection, 
while a score of '1' indicates that the marker is the most prevalent.

Start by selecting lists you want to compare.

In [2]:
results = mr.guided_search(out="marker_list")

Available columns for search:
1: List name
2: Organism name
3: Taxonomy ID
4: Marker type
5: Submitter name
6: List type
7: Date
8: Source
9: Email
a: Tissue
Enter identifier of column to search in (leave blank to search in all columns)
Enter : 1
Do you want to see all possible values for this column? (yes/no): no
Enter search terms (separate multiple terms with a comma): panglao
Perform an exact search? (yes/no): no
Consider case sensitivity? (yes/no): no
Number of results: 60
Do you want to see the results? (yes/no): no
Do you want to filter the results further? (yes/no): no


Compare the lists and show the new DataFrame.

In [3]:
results_scored = calc.compare_marker_lists(marker_df=results)
display(results_scored)

,Marker,Info,Score
8066,UNC13A ENSMUSG00000034799,Chromaffin cells,0.0
8101,HBB-BS ENSMUSG00000052305,Erythroid-like and erythroid precursor cells,0.0
8097,LMO2 ENSMUSG00000032698,Erythroblasts,0.0
8092,USE1 ENSMUSG00000002395,Erythroblasts,0.0
8091,GRSF1 ENSMUSG00000044221,Erythroblasts,0.0
...,...,...,...
8210,CXCR4 ENSMUSG00000045382,Platelets,1.0
8311,CXCR4 ENSMUSG00000045382,Hematopoietic stem cells,1.0
4390,CXCR4 ENSG00000121966,Monocytes,1.0
7219,CXCR4 ENSG00000121966,Satellite cells,1.0


With this function all marker scores, which are part of the panglaoDB, are being updated by using the calculated ubiquitousness index from the panglaoDB. 

In [4]:
results_panglao = calc.update_scores(df=results_scored)
display(results_panglao)

,Marker,Info,Score
8066,UNC13A ENSMUSG00000034799,Chromaffin cells,0.000
11279,SPRR2A3 ENSMUSG00000074445,Foveolar cells,0.000
11303,OTOGL ENSMUSG00000091455,Goblet cells,0.000
11150,TRY5 ENSMUSG00000036938,Enterocytes,0.000
11160,ADH6A ENSMUSG00000053054,Enterocytes,0.000
...,...,...,...
12964,CD81 ENSMUSG00000037706,T cells,0.598
482,PFN1 ENSG00000108518,Osteocytes,0.797
8516,PFN1 ENSMUSG00000018293,Osteocytes,0.797
10858,ITM2B ENSMUSG00000022108,Müller cells,0.854


Finally the scored list can be exported.

In [5]:
mr.export_marker_list(results_panglao)

Marker list saved: /mnt/workspace/mkessle/projects/annotate_by_marker_and_features/marker_list_20230614190159


'/mnt/workspace/mkessle/projects/annotate_by_marker_and_features/marker_list_20230614190159'

# Transfer markers using homology

Pull whitelist repository and update if necessary.

In [6]:
mr.get_whitelists()

Fetching whitelists...

Done!


Select source organism and target organism.

In [7]:
source_organism, source_tax = mr.select(key="organism", heading="source organism").split(" ")
target_organism, target_tax = mr.select(key="organism", heading="target organism").split(" ")

Select source organism
1:	human 9606
2:	mouse 10090
3:	zebrafish 7955
4:	rat 10114
5:	pig 9823
6:	medaka 8090
7:	chicken 9031
8:	drosophila 7215
9:	yeast 4932
2
Selection: mouse 10090

Select target organism
1:	human 9606
2:	mouse 10090
3:	zebrafish 7955
4:	rat 10114
5:	pig 9823
6:	medaka 8090
7:	chicken 9031
8:	drosophila 7215
9:	yeast 4932
4
Selection: rat 10114



## Transfer markers from one organism to another using BioMart

Select available source and target organism identifier from biomart db.

In [ ]:
source_organism_bm = mr.select(whitelist=calc.get_dataset_names(source_organism), heading="BioMart source organism")
target_organism_bm = mr.select(whitelist=calc.get_dataset_names(target_organism), heading="BioMart target organism")

Select BioMart source organism
1:	mspicilegus
2:	mspretus
3:	mpahari
4:	mcaroli
5:	mmurinus
6:	mmusculus
7:	pmbairdii
6
Selection: mmusculus

Select BioMart target organism
1:	dordii
2:	rnorvegicus
3:	ngalili
4:	hgfemale


Fetch necessary data from BioMart.

In [ ]:
biomart_db = calc.fetch_homologs(source_organism_bm, target_organism_bm).dropna()
biomart_db

Select all lists of source organism

In [ ]:
keywords = {"Organism name": source_organism}
source_df = mr.search_db(mr.get_db(), keywords, case_sensitive=True, exact=True, out="marker_list")
display(source_df)

Transfer source markers to target markers using BioMart results.

In [ ]:
transferred_list = merge_dataframes(biomart_db, source_df)
display(transferred_list)

Extend transferred markers, rename column

In [ ]:
gene_dict = mr.get_gene_dict(target_organism)
transferred_list.rename(columns={'Transferred Marker': 'Marker'}, inplace=True)
markers_extended = mr.update_markers(transferred_list, gene_dict)
display(markers_extended)

## Transfer markers from one organism to another using HomoloGene db

Get HomoloGene db.

In [ ]:
hg_db = calc.download_homologene_data()

In [ ]:
print(f"Supported organisms when using HomoloGene db: {calc.get_supported_taxonomy_ids(hg_db)}")

Select all lists of source organism

In [ ]:
keywords = {"Organism name": source_organism}
source_df = mr.search_db(mr.get_db(), keywords, case_sensitive=True, exact=True, out="marker_list")
display(source_df)

Transfer source markers to target markers using HomoloGene db

In [ ]:
transferred_list = calc.transfer_markers(source_df, source_tax, target_tax, hg_db)
display(transferred_list)

Extend transferred markers, rename column

In [ ]:
gene_dict = mr.get_gene_dict(target_organism)
transferred_list.rename(columns={'Transferred Marker': 'Marker'}, inplace=True)
markers_extended = mr.update_markers(transferred_list, gene_dict)
display(markers_extended)